# Sumativa 2 - Validación, simulación y métodos de remuestreo

**Proyecto:** Análisis estadístico-computacional del dataset Rain in Australia para estudiar variables meteorológicas asociadas a la ocurrencia de lluvia al día siguiente.

**Dataset:** `weatherAUS`.

**Continuidad metodológica:** este notebook utiliza como insumo directo la base procesada de la Sumativa 1 y profundiza sus resultados inferenciales mediante bootstrap, prueba de permutación, simulación Monte Carlo y análisis de robustez.

## Objetivo técnico

Validar la estabilidad estadística de los resultados obtenidos en la Sumativa 1, especialmente la diferencia de `Humidity3pm` entre los grupos `RainTomorrow = Yes` y `RainTomorrow = No`, incorporando métodos computacionales de remuestreo y simulación para preparar insumos consistentes para la Sumativa 3.

## Alcance del análisis

El desarrollo considera seis componentes integrados: auditoría de datos faltantes, recuperación de parámetros base de Sumativa 1, validación bootstrap de intervalos de confianza, contraste por permutación de la prueba de hipótesis principal, estabilidad de correlaciones mediante remuestreo, simulación Monte Carlo basada en parámetros observados y análisis de robustez frente a valores extremos, supuestos y localidades.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

cwd = Path.cwd().resolve()
if (cwd / 'semana1').exists() and (cwd / 'semana2').exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = next(p for p in [cwd, *cwd.parents] if (p / 'semana1').exists() and (p / 'semana2').exists())

SEMANA1_DIR = PROJECT_ROOT / 'semana1'
SEMANA2_DIR = PROJECT_ROOT / 'semana2'
DATA_INPUT = SEMANA2_DIR / 'data' / 'input'
DATA_IN = DATA_INPUT / 'weatherAUS_sumativa1_variables_clave.csv'
DATA_PROCESSED = SEMANA2_DIR / 'data' / 'processed'
TABLES_DIR = SEMANA2_DIR / 'docs' / 'tables'
FIGURES_DIR = SEMANA2_DIR / 'docs' / 'figures'
SRC_DIR = SEMANA2_DIR / 'src'

for directory in [DATA_INPUT, DATA_PROCESSED, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))
from remuestreo_utils import (
    normal_ci_mean, normal_ci_proportion, welch_ci_difference, cohens_d_independent,
    bootstrap_mean, bootstrap_proportion, bootstrap_difference_means,
    jackknife_mean_values, jackknife_difference_means_values,
    bca_interval, percentile_interval, permutation_difference_means,
    bootstrap_correlation, iqr_filter, winsorize_array,
    bootstrap_median_difference
)

RANDOM_SEED = 5012026
N_BOOT = 10_000
N_PERM = 10_000
N_MC = 100_000

# Configuración de remuestreo y simulación: cada procedimiento utiliza el tamaño muestral válido completo.
BOOT_SAMPLE_MEAN = None
BOOT_SAMPLE_CORR = None
BOOT_SAMPLE_PERM = None
BOOT_SAMPLE_MEDIAN = None
BATCH_SIZE_BOOTSTRAP = 20

rng = np.random.default_rng(RANDOM_SEED)

def formato_p_value(p: float, limite: float = 0.001) -> str:
    if pd.isna(p):
        return ''
    p = float(p)
    if p < limite:
        return 'p < 0,001'
    return f'p = {p:.4f}'.replace('.', ',')

print('Configuración inicial cargada correctamente.')
print('Base de entrada: semana2/data/input/weatherAUS_sumativa1_variables_clave.csv')


Configuración inicial cargada correctamente.
Base de entrada: semana2/data/input/weatherAUS_sumativa1_variables_clave.csv


## 1. Carga de datos y consistencia con Sumativa 1

La validación se desarrolla sobre la base procesada de la Sumativa 1. Se mantienen las variables meteorológicas seleccionadas, las codificaciones binarias `RainToday_bin` y `RainTomorrow_bin`, y la variable categórica `WindDir9am` para conservar trazabilidad hacia la etapa de modelamiento.

El notebook no reconstruye la limpieza exploratoria de la Semana 1. La finalidad de esta fase es validar computacionalmente los resultados ya obtenidos mediante remuestreo, permutación y simulación. Por ello, la primera tarea consiste en confirmar que la base procesada contiene las variables necesarias, conservar la trazabilidad de observaciones disponibles y establecer una regla explícita para los valores faltantes antes de aplicar los métodos de validación.


In [2]:
df = pd.read_csv(DATA_IN)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

variables_requeridas = [
    'Date', 'Location', 'Year', 'Month', 'Season', 'MinTemp', 'MaxTemp',
    'Humidity3pm', 'Pressure3pm', 'Rainfall', 'WindDir9am', 'RainToday',
    'RainTomorrow', 'RainToday_bin', 'RainTomorrow_bin'
]

faltantes = [var for var in variables_requeridas if var not in df.columns]
if faltantes:
    raise ValueError(f'Faltan variables requeridas: {faltantes}')

base_validacion = df[variables_requeridas].copy()
base_validacion.to_csv(DATA_PROCESSED / 'weatherAUS_sumativa2_base_validacion.csv', index=False)

resumen_base = pd.DataFrame({
    'indicador': [
        'registros_base_sumativa1', 'variables_base_sumativa1', 'localidades',
        'observaciones_humidity3pm_validas', 'observaciones_raintomorrow_validas',
        'proporcion_rain_tomorrow_yes'
    ],
    'valor': [
        len(base_validacion), base_validacion.shape[1], base_validacion['Location'].nunique(),
        base_validacion['Humidity3pm'].notna().sum(), base_validacion['RainTomorrow'].notna().sum(),
        base_validacion['RainTomorrow_bin'].mean()
    ]
})
resumen_base.to_csv(TABLES_DIR / '00_resumen_base_sumativa2.csv', index=False)
resumen_base


,indicador,valor
0,registros_base_sumativa1,145460.000000
1,variables_base_sumativa1,15.000000
2,localidades,49.000000
3,observaciones_humidity3pm_validas,140953.000000
4,observaciones_raintomorrow_validas,142193.000000
5,proporcion_rain_tomorrow_yes,0.224181


## 2. Auditoría de valores faltantes y decisión metodológica

El bootstrap no debe calcularse sobre valores `NaN`, porque una estadística como la media, la diferencia de medias o la correlación queda indefinida si la remuestra contiene observaciones no numéricas. Sin embargo, esto no implica que la imputación sea obligatoria en esta fase.

En la Sumativa 2 se adopta un criterio de **casos válidos por análisis**. Esto significa que cada procedimiento utiliza solamente las variables necesarias para su propia estadística. Por ejemplo, la comparación de medias de `Humidity3pm` elimina registros sin `Humidity3pm` o sin `RainTomorrow`, mientras que una correlación entre `Pressure3pm` y `RainTomorrow_bin` elimina únicamente registros incompletos en esas dos variables. Esta decisión evita descartar información útil de otras variables y mantiene coherencia con la inferencia realizada en la Sumativa 1.

No se aplica imputación en esta etapa porque el objetivo no es entrenar todavía un modelo predictivo, sino validar la estabilidad de estimadores observados. Una imputación simple antes del bootstrap podría modificar artificialmente la distribución empírica, reducir la variabilidad remuestreada y generar intervalos de confianza demasiado optimistas. 

In [3]:
auditoria_nan = (
    base_validacion
    .isna()
    .sum()
    .reset_index()
    .rename(columns={'index': 'variable', 0: 'n_faltantes'})
)
auditoria_nan['n_total'] = len(base_validacion)
auditoria_nan['porcentaje_faltante'] = 100 * auditoria_nan['n_faltantes'] / auditoria_nan['n_total']
auditoria_nan['n_validos'] = auditoria_nan['n_total'] - auditoria_nan['n_faltantes']
auditoria_nan = auditoria_nan[['variable', 'n_total', 'n_validos', 'n_faltantes', 'porcentaje_faltante']]
auditoria_nan.to_csv(TABLES_DIR / '00b_auditoria_nan_sumativa2.csv', index=False)

criterio_nan = pd.DataFrame([
    {
        'procedimiento': 'Bootstrap de media global de Humidity3pm',
        'variables_requeridas': 'Humidity3pm',
        'tratamiento_nan': 'Exclusión de registros sin Humidity3pm',
        'justificacion': 'La media remuestreada debe estimarse desde la distribución empírica observada de la variable.'
    },
    {
        'procedimiento': 'Bootstrap de medias por RainTomorrow',
        'variables_requeridas': 'Humidity3pm; RainTomorrow; RainTomorrow_bin',
        'tratamiento_nan': 'Exclusión de registros sin humedad o sin clasificación de lluvia al día siguiente',
        'justificacion': 'La diferencia de medias requiere pertenencia válida a grupo y valor numérico de humedad.'
    },
    {
        'procedimiento': 'Prueba de permutación',
        'variables_requeridas': 'Humidity3pm; RainTomorrow',
        'tratamiento_nan': 'Uso de pares completos para humedad y grupo',
        'justificacion': 'La permutación intercambia etiquetas observadas entre valores observados; las etiquetas faltantes no forman parte de la distribución nula.'
    },
    {
        'procedimiento': 'Bootstrap de correlaciones',
        'variables_requeridas': 'Variable meteorológica analizada; RainTomorrow_bin',
        'tratamiento_nan': 'Casos completos por par de variables',
        'justificacion': 'Cada correlación se estima con el máximo número de pares válidos disponibles para ese contraste.'
    },
    {
        'procedimiento': 'Simulación Monte Carlo',
        'variables_requeridas': 'Parámetros estimados desde casos válidos de Sumativa 1',
        'tratamiento_nan': 'No se simulan datos faltantes; se simulan escenarios desde parámetros observados válidos',
        'justificacion': 'La simulación evalúa escenarios probabilísticos basados en estimadores ya depurados.'
    },
    {
        'procedimiento': 'Preparación Sumativa 3',
        'variables_requeridas': 'Variables candidatas para modelamiento',
        'tratamiento_nan': 'La imputación se reserva para la fase de modelamiento predictivo',
        'justificacion': 'En modelos, la imputación debe ajustarse dentro del entrenamiento para evitar fuga de información.'
    }
])
criterio_nan.to_csv(TABLES_DIR / '00c_criterio_tratamiento_nan_sumativa2.csv', index=False)

base_humidity_groups = base_validacion.dropna(subset=['Humidity3pm', 'RainTomorrow', 'RainTomorrow_bin']).copy()
base_rain_prop = base_validacion.dropna(subset=['RainTomorrow_bin']).copy()

resumen_casos_validos = pd.DataFrame([
    {
        'analisis': 'Media global Humidity3pm',
        'variables': 'Humidity3pm',
        'n_valido': int(base_validacion['Humidity3pm'].notna().sum()),
        'n_excluido_por_nan': int(base_validacion['Humidity3pm'].isna().sum())
    },
    {
        'analisis': 'Diferencia de medias Humidity3pm por RainTomorrow',
        'variables': 'Humidity3pm; RainTomorrow; RainTomorrow_bin',
        'n_valido': int(len(base_humidity_groups)),
        'n_excluido_por_nan': int(len(base_validacion) - len(base_humidity_groups))
    },
    {
        'analisis': 'Proporción RainTomorrow = Yes',
        'variables': 'RainTomorrow_bin',
        'n_valido': int(len(base_rain_prop)),
        'n_excluido_por_nan': int(len(base_validacion) - len(base_rain_prop))
    }
])

for var in ['Humidity3pm', 'RainToday_bin', 'Rainfall', 'Pressure3pm', 'MaxTemp']:
    pair = base_validacion[[var, 'RainTomorrow_bin']].dropna()
    resumen_casos_validos = pd.concat([
        resumen_casos_validos,
        pd.DataFrame([{
            'analisis': f'Correlación {var} vs RainTomorrow_bin',
            'variables': f'{var}; RainTomorrow_bin',
            'n_valido': int(len(pair)),
            'n_excluido_por_nan': int(len(base_validacion) - len(pair))
        }])
    ], ignore_index=True)

resumen_casos_validos.to_csv(TABLES_DIR / '00d_resumen_casos_validos_por_analisis.csv', index=False)

display(auditoria_nan)
display(criterio_nan)
display(resumen_casos_validos)

,variable,n_total,n_validos,n_faltantes,porcentaje_faltante
0,Date,145460,145460,0,0.000000
1,Location,145460,145460,0,0.000000
2,Year,145460,145460,0,0.000000
3,Month,145460,145460,0,0.000000
4,Season,145460,145460,0,0.000000
5,MinTemp,145460,143975,1485,1.020899
6,MaxTemp,145460,144199,1261,0.866905
7,Humidity3pm,145460,140953,4507,3.098446
8,Pressure3pm,145460,130432,15028,10.331363
9,Rainfall,145460,142199,3261,2.241853


,procedimiento,variables_requeridas,tratamiento_nan,justificacion
0,Bootstrap de media global de Humidity3pm,Humidity3pm,Exclusión de registros sin Humidity3pm,La media remuestreada debe estimarse desde la ...
1,Bootstrap de medias por RainTomorrow,Humidity3pm; RainTomorrow; RainTomorrow_bin,Exclusión de registros sin humedad o sin clasi...,La diferencia de medias requiere pertenencia v...
2,Prueba de permutación,Humidity3pm; RainTomorrow,Uso de pares completos para humedad y grupo,La permutación intercambia etiquetas observada...
3,Bootstrap de correlaciones,Variable meteorológica analizada; RainTomorrow...,Casos completos por par de variables,Cada correlación se estima con el máximo númer...
4,Simulación Monte Carlo,Parámetros estimados desde casos válidos de Su...,No se simulan datos faltantes; se simulan esce...,La simulación evalúa escenarios probabilístico...
5,Preparación Sumativa 3,Variables candidatas para modelamiento,La imputación se reserva para la fase de model...,"En modelos, la imputación debe ajustarse dentr..."


,analisis,variables,n_valido,n_excluido_por_nan
0,Media global Humidity3pm,Humidity3pm,140953,4507
1,Diferencia de medias Humidity3pm por RainTomorrow,Humidity3pm; RainTomorrow; RainTomorrow_bin,138583,6877
2,Proporción RainTomorrow = Yes,RainTomorrow_bin,142193,3267
3,Correlación Humidity3pm vs RainTomorrow_bin,Humidity3pm; RainTomorrow_bin,138583,6877
4,Correlación RainToday_bin vs RainTomorrow_bin,RainToday_bin; RainTomorrow_bin,140787,4673
5,Correlación Rainfall vs RainTomorrow_bin,Rainfall; RainTomorrow_bin,140787,4673
6,Correlación Pressure3pm vs RainTomorrow_bin,Pressure3pm; RainTomorrow_bin,128212,17248
7,Correlación MaxTemp vs RainTomorrow_bin,MaxTemp; RainTomorrow_bin,141871,3589


## 3. Parámetros de Sumativa 1 utilizados como línea base

Se seleccionan parámetros que representan el hallazgo central de la Sumativa 1: la humedad relativa de la tarde presenta una separación estadísticamente relevante entre días sin lluvia al día siguiente y días con lluvia al día siguiente.

La selección de parámetros no se realiza de forma genérica. Cada estimador responde a un resultado ya obtenido: medias condicionales, diferencia de medias, proporción de ocurrencia de lluvia al día siguiente y asociación entre variables meteorológicas. Esta continuidad es necesaria porque la Sumativa 2 evalúa la estabilidad de resultados previos, no una exploración que no dependa de los resultados obtenidos durante la semana 1.

In [4]:
valid_humidity = base_humidity_groups.copy()

h_all_global = base_validacion['Humidity3pm'].dropna().to_numpy(dtype=float)
h_yes = valid_humidity.loc[valid_humidity['RainTomorrow'] == 'Yes', 'Humidity3pm'].to_numpy(dtype=float)
h_no = valid_humidity.loc[valid_humidity['RainTomorrow'] == 'No', 'Humidity3pm'].to_numpy(dtype=float)
h_all_grupos = valid_humidity['Humidity3pm'].to_numpy(dtype=float)
y_all_grupos = valid_humidity['RainTomorrow_bin'].to_numpy(dtype=float)

mean_all = float(np.mean(h_all_global))
sd_all = float(np.std(h_all_global, ddof=1))
mean_yes = float(np.mean(h_yes))
mean_no = float(np.mean(h_no))
diff_yes_no = mean_yes - mean_no
prop_yes = float(np.mean(base_rain_prop['RainTomorrow_bin']))
sd_yes = float(np.std(h_yes, ddof=1))
sd_no = float(np.std(h_no, ddof=1))

welch_stat = stats.ttest_ind(h_yes, h_no, equal_var=False, alternative='greater')
cohen_d = cohens_d_independent(h_yes, h_no)

parametros_s1 = pd.DataFrame([
    ['Media global de Humidity3pm', mean_all, sd_all, len(h_all_global), 'Porcentaje'],
    ['Media de Humidity3pm | RainTomorrow = No', mean_no, sd_no, len(h_no), 'Porcentaje'],
    ['Media de Humidity3pm | RainTomorrow = Yes', mean_yes, sd_yes, len(h_yes), 'Porcentaje'],
    ['Diferencia de medias Humidity3pm: Yes - No', diff_yes_no, np.nan, len(valid_humidity), 'Puntos porcentuales'],
    ['Proporción de RainTomorrow = Yes', prop_yes, np.nan, len(base_rain_prop), 'Proporción']
], columns=['parametro_s1', 'estimacion_s1', 'desviacion_estandar', 'n_valido', 'unidad'])

parametros_s1.to_csv(TABLES_DIR / '01_parametros_s1_utilizados.csv', index=False)
parametros_s1


,parametro_s1,estimacion_s1,desviacion_estandar,n_valido,unidad
0,Media global de Humidity3pm,51.539116,20.795902,140953,Porcentaje
1,Media de Humidity3pm | RainTomorrow = No,46.510625,18.489476,107670,Porcentaje
2,Media de Humidity3pm | RainTomorrow = Yes,68.800019,19.037409,30913,Porcentaje
3,Diferencia de medias Humidity3pm: Yes - No,22.289394,NaN,138583,Puntos porcentuales
4,Proporción de RainTomorrow = Yes,0.224181,NaN,142193,Proporción


## 4. Validación de parámetros mediante bootstrap no paramétrico

Se aplican 10.000 remuestras bootstrap no paramétricas. Cada remuestra utiliza el tamaño muestral válido completo del parámetro correspondiente. Esta decisión permite comparar de forma directa los intervalos clásicos de la Sumativa 1 con los intervalos bootstrap percentil y BCa calculados en esta etapa.

Para cada parámetro se calcula un intervalo de confianza clásico, un intervalo bootstrap percentil y un intervalo bootstrap BCa. El método percentil describe directamente la distribución empírica remuestreada. El método BCa incorpora corrección por sesgo y aceleración mediante valores jackknife, lo que permite contrastar si la inferencia paramétrica de la Sumativa 1 se mantiene cuando se relajan supuestos de normalidad aproximada.

La interpretación se basa en cuatro criterios: posición de la estimación puntual dentro de la distribución bootstrap, concordancia entre intervalos clásicos y bootstrap, amplitud relativa de los intervalos y consistencia del resultado con los supuestos estadísticos involucrados.


In [5]:
configuracion_remuestreo = pd.DataFrame([{
    'configuracion': 'Remuestreo y simulación con tamaño muestral válido completo',
    'n_bootstrap': N_BOOT,
    'n_permutaciones': N_PERM,
    'n_montecarlo': N_MC,
    'sample_size_bootstrap_media': 'n_valido_completo',
    'sample_size_bootstrap_correlacion': 'n_valido_completo',
    'sample_size_permutacion': 'n_valido_completo',
    'sample_size_bootstrap_mediana': 'n_valido_completo',
    'batch_size_bootstrap': BATCH_SIZE_BOOTSTRAP,
    'semilla_aleatoria': RANDOM_SEED,
    'observacion_metodologica': 'Los procedimientos se ejecutan con 10.000 réplicas para bootstrap y permutación, y 100.000 iteraciones para Monte Carlo.'
}])
configuracion_remuestreo.to_csv(TABLES_DIR / '00e_configuracion_remuestreo_sumativa2.csv', index=False)

display(configuracion_remuestreo)

bootstrap_records = []
bootstrap_series = {}

print('Bootstrap: media global')
boot_all = bootstrap_mean(h_all_global, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size=BOOT_SAMPLE_MEAN)
ci_pct = percentile_interval(boot_all)
ci_bca = bca_interval(mean_all, boot_all, jackknife_mean_values(h_all_global))
ci_classic = normal_ci_mean(h_all_global)
bootstrap_series['media_global_humidity3pm'] = boot_all
bootstrap_records.append(['Media global de Humidity3pm', mean_all, ci_classic[1], ci_classic[2], ci_pct[0], ci_pct[1], ci_bca[0], ci_bca[1], np.mean(boot_all), np.std(boot_all, ddof=1), N_BOOT])

print('Bootstrap: media grupo No')
boot_no = bootstrap_mean(h_no, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size=BOOT_SAMPLE_MEAN)
ci_pct = percentile_interval(boot_no)
ci_bca = bca_interval(mean_no, boot_no, jackknife_mean_values(h_no))
ci_classic = normal_ci_mean(h_no)
bootstrap_series['media_humidity3pm_no'] = boot_no
bootstrap_records.append(['Media de Humidity3pm | RainTomorrow = No', mean_no, ci_classic[1], ci_classic[2], ci_pct[0], ci_pct[1], ci_bca[0], ci_bca[1], np.mean(boot_no), np.std(boot_no, ddof=1), N_BOOT])

print('Bootstrap: media grupo Yes')
boot_yes = bootstrap_mean(h_yes, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size=BOOT_SAMPLE_MEAN)
ci_pct = percentile_interval(boot_yes)
ci_bca = bca_interval(mean_yes, boot_yes, jackknife_mean_values(h_yes))
ci_classic = normal_ci_mean(h_yes)
bootstrap_series['media_humidity3pm_yes'] = boot_yes
bootstrap_records.append(['Media de Humidity3pm | RainTomorrow = Yes', mean_yes, ci_classic[1], ci_classic[2], ci_pct[0], ci_pct[1], ci_bca[0], ci_bca[1], np.mean(boot_yes), np.std(boot_yes, ddof=1), N_BOOT])

print('Bootstrap: diferencia de medias')
boot_diff = bootstrap_difference_means(h_yes, h_no, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size_yes=BOOT_SAMPLE_MEAN, sample_size_no=BOOT_SAMPLE_MEAN)
ci_pct = percentile_interval(boot_diff)
ci_bca = bca_interval(diff_yes_no, boot_diff, jackknife_difference_means_values(h_yes, h_no))
ci_classic = welch_ci_difference(h_yes, h_no)
bootstrap_series['diferencia_medias_humidity3pm_yes_no'] = boot_diff
bootstrap_records.append(['Diferencia de medias Humidity3pm: Yes - No', diff_yes_no, ci_classic[1], ci_classic[2], ci_pct[0], ci_pct[1], ci_bca[0], ci_bca[1], np.mean(boot_diff), np.std(boot_diff, ddof=1), N_BOOT])

print('Bootstrap: proporción RainTomorrow Yes')
rain_tomorrow_valid = base_validacion['RainTomorrow_bin'].dropna().to_numpy(dtype=float)
boot_prop = bootstrap_proportion(rain_tomorrow_valid, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size=BOOT_SAMPLE_MEAN)
ci_pct = percentile_interval(boot_prop)
ci_bca = bca_interval(prop_yes, boot_prop, jackknife_mean_values(rain_tomorrow_valid))
ci_classic = normal_ci_proportion(rain_tomorrow_valid)
bootstrap_series['proporcion_raintomorrow_yes'] = boot_prop
bootstrap_records.append(['Proporción de RainTomorrow = Yes', prop_yes, ci_classic[1], ci_classic[2], ci_pct[0], ci_pct[1], ci_bca[0], ci_bca[1], np.mean(boot_prop), np.std(boot_prop, ddof=1), N_BOOT])

bootstrap_parametros = pd.DataFrame(bootstrap_records, columns=[
    'parametro', 'estimacion_s1', 'ic_clasico_95_li', 'ic_clasico_95_ls',
    'ic_percentil_95_li', 'ic_percentil_95_ls', 'ic_bca_95_li', 'ic_bca_95_ls',
    'media_bootstrap', 'error_estandar_bootstrap', 'n_remuestras'
])

bootstrap_parametros['ancho_ic_clasico'] = bootstrap_parametros['ic_clasico_95_ls'] - bootstrap_parametros['ic_clasico_95_li']
bootstrap_parametros['ancho_ic_percentil'] = bootstrap_parametros['ic_percentil_95_ls'] - bootstrap_parametros['ic_percentil_95_li']
bootstrap_parametros['ancho_ic_bca'] = bootstrap_parametros['ic_bca_95_ls'] - bootstrap_parametros['ic_bca_95_li']
bootstrap_parametros.to_csv(TABLES_DIR / '02_bootstrap_parametros_s1.csv', index=False)

comparacion_intervalos = bootstrap_parametros.copy()
comparacion_intervalos['razon_ancho_percentil_vs_clasico'] = comparacion_intervalos['ancho_ic_percentil'] / comparacion_intervalos['ancho_ic_clasico']
comparacion_intervalos['razon_ancho_bca_vs_clasico'] = comparacion_intervalos['ancho_ic_bca'] / comparacion_intervalos['ancho_ic_clasico']
comparacion_intervalos['incluye_estimacion_s1_percentil'] = (
    (comparacion_intervalos['estimacion_s1'] >= comparacion_intervalos['ic_percentil_95_li']) &
    (comparacion_intervalos['estimacion_s1'] <= comparacion_intervalos['ic_percentil_95_ls'])
)
comparacion_intervalos['incluye_estimacion_s1_bca'] = (
    (comparacion_intervalos['estimacion_s1'] >= comparacion_intervalos['ic_bca_95_li']) &
    (comparacion_intervalos['estimacion_s1'] <= comparacion_intervalos['ic_bca_95_ls'])
)
comparacion_intervalos['lectura_metodologica'] = np.where(
    comparacion_intervalos['incluye_estimacion_s1_bca'],
    'El intervalo BCa contiene la estimación de S1; la evidencia bootstrap es concordante con el resultado previo.',
    'El intervalo BCa no contiene la estimación de S1; revisar posible sesgo, asimetría o sensibilidad del estimador.'
)
comparacion_intervalos.to_csv(TABLES_DIR / '02b_comparacion_intervalos_bootstrap.csv', index=False)

bootstrap_parametros


,configuracion,n_bootstrap,n_permutaciones,n_montecarlo,sample_size_bootstrap_media,sample_size_bootstrap_correlacion,sample_size_permutacion,sample_size_bootstrap_mediana,batch_size_bootstrap,semilla_aleatoria,observacion_metodologica
0,Remuestreo y simulación con tamaño muestral vá...,10000,10000,100000,n_valido_completo,n_valido_completo,n_valido_completo,n_valido_completo,20,5012026,Los procedimientos se ejecutan con 10.000 répl...


Bootstrap: media global
Bootstrap: media grupo No
Bootstrap: media grupo Yes
Bootstrap: diferencia de medias
Bootstrap: proporción RainTomorrow Yes


,parametro,estimacion_s1,ic_clasico_95_li,ic_clasico_95_ls,ic_percentil_95_li,ic_percentil_95_ls,ic_bca_95_li,ic_bca_95_ls,media_bootstrap,error_estandar_bootstrap,n_remuestras,ancho_ic_clasico,ancho_ic_percentil,ancho_ic_bca
0,Media global de Humidity3pm,51.539116,51.430550,51.647682,51.430374,51.648830,51.430267,51.648622,51.539126,0.055528,10000,0.217131,0.218456,0.218356
1,Media de Humidity3pm | RainTomorrow = No,46.510625,46.400184,46.621066,46.400490,46.623352,46.400582,46.623437,46.510771,0.056564,10000,0.220882,0.222862,0.222854
2,Media de Humidity3pm | RainTomorrow = Yes,68.800019,68.587792,69.012247,68.591787,69.013857,68.592818,69.014735,68.800106,0.107863,10000,0.424456,0.422070,0.421917
3,Diferencia de medias Humidity3pm: Yes - No,22.289394,22.050152,22.528637,22.050744,22.525766,22.058372,22.531655,22.287794,0.121044,10000,0.478485,0.475022,0.473284
4,Proporción de RainTomorrow = Yes,0.224181,0.222014,0.226349,0.222078,0.226361,0.222085,0.226361,0.224190,0.001110,10000,0.004335,0.004283,0.004276


In [6]:
fig_specs = [
    ('media_global_humidity3pm', 'Media global de Humidity3pm', mean_all, 'fig_01_bootstrap_media_global_humidity3pm.png'),
    ('media_humidity3pm_no', 'Media Humidity3pm | RainTomorrow = No', mean_no, 'fig_02_bootstrap_media_humidity3pm_no.png'),
    ('media_humidity3pm_yes', 'Media Humidity3pm | RainTomorrow = Yes', mean_yes, 'fig_03_bootstrap_media_humidity3pm_yes.png'),
    ('diferencia_medias_humidity3pm_yes_no', 'Diferencia de medias Humidity3pm: Yes - No', diff_yes_no, 'fig_04_bootstrap_diferencia_medias.png'),
    ('proporcion_raintomorrow_yes', 'Proporción RainTomorrow = Yes', prop_yes, 'fig_05_bootstrap_proporcion_raintomorrow_yes.png')
]

for key, title, obs, filename in fig_specs:
    values = bootstrap_series[key]
    plt.figure(figsize=(7, 4.2))
    plt.hist(values, bins=45, edgecolor='black', alpha=0.75)
    plt.axvline(obs, linestyle='--', linewidth=2, label='Estimación S1')
    plt.xlabel('Estimación bootstrap')
    plt.ylabel('Frecuencia')
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=220)
    plt.close()

## 5. Validación de la prueba de hipótesis mediante permutación

La Sumativa 1 utilizó una prueba t de Welch unilateral para contrastar si la media de `Humidity3pm` es mayor cuando `RainTomorrow = Yes`. En esta etapa se valida ese resultado mediante permutación de etiquetas, generando una distribución nula empírica de la diferencia de medias.

El contraste por permutación mantiene los valores observados de humedad y rompe aleatoriamente la asociación entre humedad y grupo. Bajo la hipótesis nula, la etiqueta `RainTomorrow` no debería modificar sistemáticamente la media de `Humidity3pm`. Por lo tanto, si la diferencia observada queda en la cola extrema de la distribución nula, el resultado de la prueba t de Welch queda respaldado por un método no paramétrico.

Los registros con `NaN` no se imputan en este procedimiento, porque la permutación requiere valores observados y etiquetas observadas. Imputar antes de permutar mezclaría datos reales y datos estimados dentro de la distribución nula.

In [7]:
print('Permutación: diferencia de medias')
pooled_humidity = np.concatenate([h_yes, h_no])
perm_diffs = permutation_difference_means(pooled_humidity, n_yes=len(h_yes), n_perm=N_PERM, rng=rng, sample_size=BOOT_SAMPLE_PERM)
p_perm_one_sided = (np.sum(perm_diffs >= diff_yes_no) + 1) / (N_PERM + 1)
p_perm_two_sided = (np.sum(np.abs(perm_diffs) >= abs(diff_yes_no)) + 1) / (N_PERM + 1)

concordancia_prueba = (
    'Concordante: Welch y permutación rechazan H0 al 5 %.'
    if float(welch_stat.pvalue) < 0.05 and p_perm_one_sided < 0.05
    else 'No concordante: revisar supuestos, tamaño de efecto y distribución nula.'
)

permutacion_resultado = pd.DataFrame([{
    'prueba_validada_s1': 't de Welch unilateral para Humidity3pm según RainTomorrow',
    'hipotesis_nula': 'Media Humidity3pm Yes = Media Humidity3pm No',
    'hipotesis_alternativa': 'Media Humidity3pm Yes > Media Humidity3pm No',
    'diferencia_observada_yes_no': diff_yes_no,
    't_welch_s1': float(welch_stat.statistic),
    'p_value_parametrico_s1': float(welch_stat.pvalue),
    'p_value_parametrico_s1_formato': formato_p_value(float(welch_stat.pvalue)),
    'p_value_permutacion_unilateral': float(p_perm_one_sided),
    'p_value_permutacion_unilateral_formato': formato_p_value(float(p_perm_one_sided)),
    'p_value_permutacion_bilateral': float(p_perm_two_sided),
    'p_value_permutacion_bilateral_formato': formato_p_value(float(p_perm_two_sided)),
    'n_permutaciones': N_PERM,
    'sample_size_permutacion': 'n_valido_completo',
    'media_distribucion_nula': float(np.mean(perm_diffs)),
    'sd_distribucion_nula': float(np.std(perm_diffs, ddof=1)),
    'q025_nula': float(np.quantile(perm_diffs, 0.025)),
    'q975_nula': float(np.quantile(perm_diffs, 0.975)),
    'concordancia_parametrico_permutacion': concordancia_prueba,
    'discusion_metodologica': 'La permutación contrasta la diferencia observada contra una distribución nula empírica obtenida al reordenar etiquetas, por lo que complementa a Welch cuando se desea reducir dependencia de supuestos paramétricos.'
}])
permutacion_resultado.to_csv(TABLES_DIR / '03_test_permutacion_humidity3pm.csv', index=False)

plt.figure(figsize=(7, 4.2))
plt.hist(perm_diffs, bins=55, edgecolor='black', alpha=0.75)
plt.axvline(diff_yes_no, linestyle='--', linewidth=2, label='Diferencia observada S1')
plt.xlabel('Diferencia de medias bajo H0')
plt.ylabel('Frecuencia')
plt.title('Distribución nula por permutación')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_06_permutacion_diferencia_medias.png', dpi=220)
plt.close()

permutacion_resultado


Permutación: diferencia de medias


,prueba_validada_s1,hipotesis_nula,hipotesis_alternativa,diferencia_observada_yes_no,t_welch_s1,p_value_parametrico_s1,p_value_parametrico_s1_formato,p_value_permutacion_unilateral,p_value_permutacion_unilateral_formato,p_value_permutacion_bilateral,p_value_permutacion_bilateral_formato,n_permutaciones,sample_size_permutacion,media_distribucion_nula,sd_distribucion_nula,q025_nula,q975_nula,concordancia_parametrico_permutacion,discusion_metodologica
0,t de Welch unilateral para Humidity3pm según R...,Media Humidity3pm Yes = Media Humidity3pm No,Media Humidity3pm Yes > Media Humidity3pm No,22.289394,182.607693,0.0,"p < 0,001",0.0001,"p < 0,001",0.0001,"p < 0,001",10000,n_valido_completo,-0.00085,0.132733,-0.260515,0.258572,Concordante: Welch y permutación rechazan H0 a...,La permutación contrasta la diferencia observa...


## 6. Estabilidad de correlaciones mediante bootstrap

Se evalúan correlaciones relevantes de la Sumativa 1 entre variables meteorológicas y `RainTomorrow_bin`. La estabilidad se analiza a partir de tres elementos: dirección del coeficiente, intervalo bootstrap al 95 % y forma de la distribución bootstrap del estadístico.

Cada correlación utiliza casos completos solamente para el par de variables analizado. Esta decisión es importante porque las variables meteorológicas no tienen necesariamente el mismo patrón de datos faltantes. Una correlación se considera estable cuando su intervalo bootstrap al 95 % no incluye el cero y mantiene una dirección compatible con el resultado estimado previamente.


In [8]:
correlation_vars = ['Humidity3pm', 'RainToday_bin', 'Rainfall', 'Pressure3pm', 'MaxTemp']
correlation_records = []
correlation_bootstrap_store = {}

for var in correlation_vars:
    print(f'Bootstrap correlación: {var}')
    pair = base_validacion[[var, 'RainTomorrow_bin']].dropna()
    x = pair[var].to_numpy(dtype=float)
    y = pair['RainTomorrow_bin'].to_numpy(dtype=float)
    r_obs = float(np.corrcoef(x, y)[0, 1])
    boot_r = bootstrap_correlation(x, y, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP, sample_size=BOOT_SAMPLE_CORR)
    li, ls = percentile_interval(boot_r)
    ancho = ls - li
    incluye_cero = bool(li <= 0 <= ls)
    stable = not incluye_cero
    if ancho <= 0.05:
        variabilidad = 'baja'
    elif ancho <= 0.10:
        variabilidad = 'moderada'
    else:
        variabilidad = 'alta'
    if stable:
        diagnostico = f'Estable: el IC bootstrap 95 % no incluye cero y conserva dirección {"positiva" if r_obs > 0 else "negativa"}.'
    else:
        diagnostico = 'Requiere cautela: el IC bootstrap 95 % incluye cero o no sostiene dirección definida.'
    correlation_bootstrap_store[var] = boot_r
    correlation_records.append({
        'variable': var,
        'variable_objetivo': 'RainTomorrow_bin',
        'correlacion_pearson_s1': r_obs,
        'ic_bootstrap_95_li': li,
        'ic_bootstrap_95_ls': ls,
        'ancho_ic_bootstrap': ancho,
        'n_pares_validos': len(pair),
        'n_remuestras': N_BOOT,
        'incluye_cero_95': incluye_cero,
        'variabilidad_bootstrap': variabilidad,
        'estable_95': stable,
        'diagnostico_correlacion': diagnostico
    })

correlaciones_bootstrap = pd.DataFrame(correlation_records).sort_values('correlacion_pearson_s1', key=lambda s: s.abs(), ascending=False)
correlaciones_bootstrap.to_csv(TABLES_DIR / '04_bootstrap_correlaciones.csv', index=False)

estabilidad_correlaciones = correlaciones_bootstrap[[
    'variable', 'variable_objetivo', 'correlacion_pearson_s1',
    'ic_bootstrap_95_li', 'ic_bootstrap_95_ls', 'ancho_ic_bootstrap',
    'incluye_cero_95', 'variabilidad_bootstrap', 'estable_95', 'diagnostico_correlacion'
]].copy()
estabilidad_correlaciones.to_csv(TABLES_DIR / '04b_diagnostico_estabilidad_correlaciones.csv', index=False)

plt.figure(figsize=(7, 4.2))
y_pos = np.arange(len(correlaciones_bootstrap))
mid = correlaciones_bootstrap['correlacion_pearson_s1'].to_numpy()
err_low = mid - correlaciones_bootstrap['ic_bootstrap_95_li'].to_numpy()
err_high = correlaciones_bootstrap['ic_bootstrap_95_ls'].to_numpy() - mid
plt.errorbar(mid, y_pos, xerr=[err_low, err_high], fmt='o', capsize=4)
plt.axvline(0, linestyle='--', linewidth=1)
plt.yticks(y_pos, correlaciones_bootstrap['variable'])
plt.xlabel('Correlación de Pearson con RainTomorrow_bin')
plt.title('Intervalos bootstrap al 95 % para correlaciones')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_07_bootstrap_correlaciones.png', dpi=220)
plt.close()

fig, axes = plt.subplots(len(correlaciones_bootstrap), 1, figsize=(7, 10), sharex=False)
for ax, (_, row) in zip(axes, correlaciones_bootstrap.iterrows()):
    var = row['variable']
    boot_values = correlation_bootstrap_store[var]
    ax.hist(boot_values, bins=40, density=True, alpha=0.85)
    ax.axvline(row['correlacion_pearson_s1'], linewidth=1.5, label='Correlación observada')
    ax.axvline(row['ic_bootstrap_95_li'], linestyle='--', linewidth=1, label='IC 95 %')
    ax.axvline(row['ic_bootstrap_95_ls'], linestyle='--', linewidth=1)
    ax.axvline(0, linestyle=':', linewidth=1)
    ax.set_title(f'{var} vs RainTomorrow_bin')
    ax.set_xlabel('Correlación bootstrap')
    ax.set_ylabel('Densidad')
    ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_07b_distribuciones_bootstrap_correlaciones.png', dpi=220)
plt.close()

correlaciones_bootstrap


Bootstrap correlación: Humidity3pm
Bootstrap correlación: RainToday_bin
Bootstrap correlación: Rainfall
Bootstrap correlación: Pressure3pm
Bootstrap correlación: MaxTemp


,variable,variable_objetivo,correlacion_pearson_s1,ic_bootstrap_95_li,ic_bootstrap_95_ls,ancho_ic_bootstrap,n_pares_validos,n_remuestras,incluye_cero_95,variabilidad_bootstrap,estable_95,diagnostico_correlacion
0,Humidity3pm,RainTomorrow_bin,0.446160,0.441827,0.450565,0.008738,138583,10000,False,baja,True,Estable: el IC bootstrap 95 % no incluye cero ...
1,RainToday_bin,RainTomorrow_bin,0.313097,0.307411,0.318717,0.011306,140787,10000,False,baja,True,Estable: el IC bootstrap 95 % no incluye cero ...
2,Rainfall,RainTomorrow_bin,0.239032,0.232009,0.246315,0.014306,140787,10000,False,baja,True,Estable: el IC bootstrap 95 % no incluye cero ...
3,Pressure3pm,RainTomorrow_bin,-0.226031,-0.231297,-0.220649,0.010649,128212,10000,False,baja,True,Estable: el IC bootstrap 95 % no incluye cero ...
4,MaxTemp,RainTomorrow_bin,-0.159237,-0.164195,-0.154210,0.009985,141871,10000,False,baja,True,Estable: el IC bootstrap 95 % no incluye cero ...


## 7. Simulación Monte Carlo basada en parámetros de Sumativa 1

La simulación utiliza exclusivamente parámetros estimados desde casos válidos de la Sumativa 1: medias y desviaciones estándar de `Humidity3pm` por grupo, proporción de `RainTomorrow = Yes` y estructura binaria del evento de lluvia. El objetivo es traducir el hallazgo inferencial a escenarios probabilísticos útiles para la preparación de un modelo predictivo posterior.

La variable simulada `delta_individual_yes_no` representa la diferencia entre dos observaciones simuladas independientes: una bajo los parámetros del grupo `RainTomorrow = Yes` y otra bajo los parámetros del grupo `RainTomorrow = No`. Por lo tanto, no corresponde a un intervalo inferencial de la diferencia de medias. La incertidumbre inferencial de la diferencia de medias se evalúa previamente mediante Welch, bootstrap percentil, bootstrap BCa y permutación.

Los umbrales de `Humidity3pm` se interpretan como reglas exploratorias de alerta. No constituyen reglas finales de clasificación ni reemplazan un modelo multivariable con validación cruzada.


In [9]:
print('Monte Carlo: escenarios simulados')
mc_rng = np.random.default_rng(RANDOM_SEED + 777)

rain_event = mc_rng.binomial(1, prop_yes, size=N_MC)
humidity_yes = mc_rng.normal(mean_yes, sd_yes, size=N_MC)
humidity_no = mc_rng.normal(mean_no, sd_no, size=N_MC)
humidity_sim = np.where(rain_event == 1, humidity_yes, humidity_no)
humidity_sim = np.clip(humidity_sim, 0, 100)

delta_individual_yes_no = (
    np.clip(mc_rng.normal(mean_yes, sd_yes, size=N_MC), 0, 100)
    - np.clip(mc_rng.normal(mean_no, sd_no, size=N_MC), 0, 100)
)

thresholds = [50, 60, 70, 80]
threshold_records = []
for threshold in thresholds:
    alert = humidity_sim >= threshold
    tp = int(np.sum((alert == 1) & (rain_event == 1)))
    fp = int(np.sum((alert == 1) & (rain_event == 0)))
    tn = int(np.sum((alert == 0) & (rain_event == 0)))
    fn = int(np.sum((alert == 0) & (rain_event == 1)))
    threshold_records.append({
        'umbral_humidity3pm': threshold,
        'sensibilidad': tp / (tp + fn) if (tp + fn) else np.nan,
        'especificidad': tn / (tn + fp) if (tn + fp) else np.nan,
        'valor_predictivo_positivo': tp / (tp + fp) if (tp + fp) else np.nan,
        'valor_predictivo_negativo': tn / (tn + fn) if (tn + fn) else np.nan,
        'proporcion_alertas': float(np.mean(alert)),
        'probabilidad_rain_yes_condicional_alerta': tp / (tp + fp) if (tp + fp) else np.nan
    })

montecarlo_resumen = pd.DataFrame([{
    'n_iteraciones': N_MC,
    'media_delta_individual_yes_no': float(np.mean(delta_individual_yes_no)),
    'mediana_delta_individual_yes_no': float(np.median(delta_individual_yes_no)),
    'sd_delta_individual_yes_no': float(np.std(delta_individual_yes_no, ddof=1)),
    'ic_empirico_95_li_delta_individual': float(np.quantile(delta_individual_yes_no, 0.025)),
    'ic_empirico_95_ls_delta_individual': float(np.quantile(delta_individual_yes_no, 0.975)),
    'probabilidad_delta_individual_mayor_0': float(np.mean(delta_individual_yes_no > 0)),
    'probabilidad_delta_individual_mayor_10': float(np.mean(delta_individual_yes_no > 10)),
    'probabilidad_delta_individual_mayor_20': float(np.mean(delta_individual_yes_no > 20)),
    'media_humidity_simulada': float(np.mean(humidity_sim)),
    'proporcion_lluvia_simulada': float(np.mean(rain_event)),
    'nota_metodologica': 'delta_individual_yes_no compara observaciones simuladas independientes, no diferencias de medias remuestreadas'
}])

montecarlo_umbral = pd.DataFrame(threshold_records)
montecarlo_resumen.to_csv(TABLES_DIR / '05_resultados_montecarlo_resumen.csv', index=False)
montecarlo_umbral.to_csv(TABLES_DIR / '06_resultados_montecarlo_umbrales.csv', index=False)

plt.figure(figsize=(7, 4.2))
plt.hist(delta_individual_yes_no, bins=60, edgecolor='black', alpha=0.75)
plt.axvline(0, linestyle='--', linewidth=1.8, label='Sin diferencia')
plt.axvline(np.mean(delta_individual_yes_no), linestyle='-', linewidth=2, label='Media simulada')
plt.xlabel('Delta individual simulado Humidity3pm Yes - No')
plt.ylabel('Frecuencia')
plt.title('Simulación Monte Carlo de delta individual de humedad')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_08_montecarlo_distribucion_delta.png', dpi=220)
plt.close()

convergencia = pd.DataFrame({
    'iteracion': np.arange(1, N_MC + 1),
    'media_acumulada_delta_individual': np.cumsum(delta_individual_yes_no) / np.arange(1, N_MC + 1)
})
convergencia_puntos = convergencia.iloc[np.unique(np.r_[0:1000:25, 1000:10000:250, 10000:N_MC:2500, N_MC - 1])].copy()
convergencia_puntos.to_csv(TABLES_DIR / '07_convergencia_montecarlo.csv', index=False)

plt.figure(figsize=(7, 4.2))
plt.plot(convergencia_puntos['iteracion'], convergencia_puntos['media_acumulada_delta_individual'])
plt.axhline(np.mean(delta_individual_yes_no), linestyle='--', linewidth=1.5, label='Media final')
plt.xlabel('Iteración')
plt.ylabel('Media acumulada del delta individual')
plt.title('Convergencia Monte Carlo')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_09_convergencia_montecarlo.png', dpi=220)
plt.close()

montecarlo_resumen, montecarlo_umbral


Monte Carlo: escenarios simulados


(   n_iteraciones  media_delta_individual_yes_no  \
 0         100000                      21.783047   
 
    mediana_delta_individual_yes_no  sd_delta_individual_yes_no  \
 0                        22.141109                   25.788127   
 
    ic_empirico_95_li_delta_individual  ic_empirico_95_ls_delta_individual  \
 0                          -29.484745                           71.068816   
 
    probabilidad_delta_individual_mayor_0  \
 0                                  0.799   
 
    probabilidad_delta_individual_mayor_10  \
 0                                 0.67838   
 
    probabilidad_delta_individual_mayor_20  media_humidity_simulada  \
 0                                 0.53175                51.455383   
 
    proporcion_lluvia_simulada  \
 0                     0.22397   
 
                                    nota_metodologica  
 0  delta_individual_yes_no compara observaciones ...  ,
    umbral_humidity3pm  sensibilidad  especificidad  valor_predictivo_positivo  \
 0   

## 8. Análisis de robustez

La robustez se evalúa desde cuatro perspectivas: eliminación de valores extremos por regla IQR, winsorización al 1 % y 99 %, diferencia de medianas mediante bootstrap y sensibilidad por localidad. Este análisis permite distinguir conclusiones confiables de conclusiones que podrían depender de supuestos paramétricos o de subconjuntos influyentes.

El tratamiento de valores extremos se diferencia del tratamiento de valores faltantes. Los valores `NaN` no contienen información numérica para el estimador y se excluyen por análisis. Los valores extremos, en cambio, sí son observaciones reales; por eso no se eliminan de forma automática, sino que se evalúa si el hallazgo principal permanece bajo reglas alternativas de sensibilidad.

In [10]:
print('Robustez: escenarios de sensibilidad')
robust_records = []

robust_records.append({
    'escenario': 'Original',
    'tipo_estimador': 'media',
    'n_yes': len(h_yes),
    'n_no': len(h_no),
    'estimador_yes': mean_yes,
    'estimador_no': mean_no,
    'diferencia_estimador_yes_no': diff_yes_no,
    'cohens_d': cohen_d,
    'p_value_welch_unilateral': float(welch_stat.pvalue),
    'p_value_welch_formato': formato_p_value(float(welch_stat.pvalue))
})

h_yes_iqr = iqr_filter(h_yes)
h_no_iqr = iqr_filter(h_no)
t_iqr = stats.ttest_ind(h_yes_iqr, h_no_iqr, equal_var=False, alternative='greater')
robust_records.append({
    'escenario': 'Filtro IQR por grupo',
    'tipo_estimador': 'media',
    'n_yes': len(h_yes_iqr),
    'n_no': len(h_no_iqr),
    'estimador_yes': float(np.mean(h_yes_iqr)),
    'estimador_no': float(np.mean(h_no_iqr)),
    'diferencia_estimador_yes_no': float(np.mean(h_yes_iqr) - np.mean(h_no_iqr)),
    'cohens_d': cohens_d_independent(h_yes_iqr, h_no_iqr),
    'p_value_welch_unilateral': float(t_iqr.pvalue),
    'p_value_welch_formato': formato_p_value(float(t_iqr.pvalue))
})

h_yes_win = winsorize_array(h_yes, 0.01, 0.99)
h_no_win = winsorize_array(h_no, 0.01, 0.99)
t_win = stats.ttest_ind(h_yes_win, h_no_win, equal_var=False, alternative='greater')
robust_records.append({
    'escenario': 'Winsorización 1 %-99 % por grupo',
    'tipo_estimador': 'media',
    'n_yes': len(h_yes_win),
    'n_no': len(h_no_win),
    'estimador_yes': float(np.mean(h_yes_win)),
    'estimador_no': float(np.mean(h_no_win)),
    'diferencia_estimador_yes_no': float(np.mean(h_yes_win) - np.mean(h_no_win)),
    'cohens_d': cohens_d_independent(h_yes_win, h_no_win),
    'p_value_welch_unilateral': float(t_win.pvalue),
    'p_value_welch_formato': formato_p_value(float(t_win.pvalue))
})

median_yes = float(np.median(h_yes))
median_no = float(np.median(h_no))
boot_median_diff = bootstrap_median_difference(
    h_yes, h_no, N_BOOT, rng, batch_size=BATCH_SIZE_BOOTSTRAP,
    sample_size_yes=BOOT_SAMPLE_MEDIAN, sample_size_no=BOOT_SAMPLE_MEDIAN
)
li_med, ls_med = percentile_interval(boot_median_diff)
robust_records.append({
    'escenario': 'Diferencia de medianas bootstrap',
    'tipo_estimador': 'mediana',
    'n_yes': len(h_yes),
    'n_no': len(h_no),
    'estimador_yes': median_yes,
    'estimador_no': median_no,
    'diferencia_estimador_yes_no': median_yes - median_no,
    'cohens_d': np.nan,
    'p_value_welch_unilateral': np.nan,
    'p_value_welch_formato': ''
})

robustez = pd.DataFrame(robust_records)
robustez['ic95_li_diferencia_mediana'] = np.nan
robustez['ic95_ls_diferencia_mediana'] = np.nan
robustez.loc[robustez['escenario'] == 'Diferencia de medianas bootstrap', 'ic95_li_diferencia_mediana'] = li_med
robustez.loc[robustez['escenario'] == 'Diferencia de medianas bootstrap', 'ic95_ls_diferencia_mediana'] = ls_med
robustez.to_csv(TABLES_DIR / '08_robustez_outliers_supuestos.csv', index=False)

location_records = []
for loc in sorted(valid_humidity['Location'].dropna().unique()):
    sub = valid_humidity[valid_humidity['Location'] != loc]
    sub_yes = sub.loc[sub['RainTomorrow'] == 'Yes', 'Humidity3pm'].to_numpy(dtype=float)
    sub_no = sub.loc[sub['RainTomorrow'] == 'No', 'Humidity3pm'].to_numpy(dtype=float)
    if len(sub_yes) > 1 and len(sub_no) > 1:
        dloc = float(np.mean(sub_yes) - np.mean(sub_no))
        location_records.append({
            'localidad_excluida': loc,
            'diferencia_media_yes_no_sin_localidad': dloc,
            'variacion_respecto_original': dloc - diff_yes_no,
            'n_restante': len(sub)
        })

sensibilidad_localidad = pd.DataFrame(location_records).sort_values('variacion_respecto_original', key=lambda s: s.abs(), ascending=False)
sensibilidad_localidad.to_csv(TABLES_DIR / '09_sensibilidad_por_localidad.csv', index=False)

max_sensibilidad_localidad = sensibilidad_localidad.iloc[0] if not sensibilidad_localidad.empty else None
sintesis_robustez = pd.DataFrame([
    {
        'dimension_evaluada': 'Valores extremos',
        'metodo': 'Filtro IQR por grupo',
        'resultado': f'Diferencia = {robustez.loc[robustez.escenario == "Filtro IQR por grupo", "diferencia_estimador_yes_no"].iloc[0]:.4f}',
        'interpretacion': 'El resultado se mantiene positivo y de magnitud alta después de excluir extremos por grupo.'
    },
    {
        'dimension_evaluada': 'Valores extremos',
        'metodo': 'Winsorización 1 %-99 %',
        'resultado': f'Diferencia = {robustez.loc[robustez.escenario == "Winsorización 1 %-99 % por grupo", "diferencia_estimador_yes_no"].iloc[0]:.4f}',
        'interpretacion': 'El resultado se mantiene positivo después de limitar valores extremos, lo que respalda estabilidad del estimador.'
    },
    {
        'dimension_evaluada': 'Métrica del centro',
        'metodo': 'Diferencia de medianas con bootstrap',
        'resultado': f'Diferencia de medianas = {median_yes - median_no:.4f}; IC 95 % [{li_med:.4f}, {ls_med:.4f}]',
        'interpretacion': 'La separación entre grupos persiste al usar una métrica robusta frente a asimetría y extremos.'
    },
    {
        'dimension_evaluada': 'Subconjuntos influyentes',
        'metodo': 'Exclusión de una localidad a la vez',
        'resultado': 'Sin sensibilidad crítica' if max_sensibilidad_localidad is None else f'Mayor variación: {max_sensibilidad_localidad["localidad_excluida"]} = {max_sensibilidad_localidad["variacion_respecto_original"]:.4f}',
        'interpretacion': 'La conclusión principal no depende de una única localidad si la diferencia mantiene signo y magnitud comparables.'
    }
])
sintesis_robustez.to_csv(TABLES_DIR / '09b_sintesis_robustez.csv', index=False)


plt.figure(figsize=(7, 4.2))
plt.barh(robustez['escenario'], robustez['diferencia_estimador_yes_no'])
plt.axvline(diff_yes_no, linestyle='--', linewidth=1.5, label='Diferencia original de medias')
plt.xlabel('Diferencia del estimador Humidity3pm Yes - No')
plt.title('Robustez de la diferencia entre grupos')
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_10_robustez_diferencia_medias.png', dpi=220)
plt.close()

robustez, sensibilidad_localidad.head(10)


Robustez: escenarios de sensibilidad


(                          escenario tipo_estimador  n_yes    n_no  \
 0                          Original          media  30913  107670   
 1              Filtro IQR por grupo          media  30694  107670   
 2  Winsorización 1 %-99 % por grupo          media  30913  107670   
 3  Diferencia de medianas bootstrap        mediana  30913  107670   
 
    estimador_yes  estimador_no  diferencia_estimador_yes_no  cohens_d  \
 0      68.800019     46.510625                    22.289394  1.197511   
 1      69.201310     46.510625                    22.690685  1.227085   
 2      68.849416     46.479474                    22.369942  1.212991   
 3      70.000000     47.000000                    23.000000       NaN   
 
    p_value_welch_unilateral p_value_welch_formato  ic95_li_diferencia_mediana  \
 0                       0.0             p < 0,001                         NaN   
 1                       0.0             p < 0,001                         NaN   
 2                       0.0  

## 9. Resultados validados para Sumativa 3

Los resultados validados se utilizarán como insumos para modelamiento predictivo posterior. Esta sección documenta parámetros robustamente estimados, correlaciones estables, sensibilidad frente a valores extremos y posibles subconjuntos influyentes. La prioridad metodológica se asigna a variables con estabilidad por remuestreo, interpretación física y bajo riesgo de dependencia exclusiva de supuestos paramétricos.


In [11]:
rainfall_r = correlaciones_bootstrap.loc[correlaciones_bootstrap.variable == 'Rainfall', 'correlacion_pearson_s1'].iloc[0]
pressure_r = correlaciones_bootstrap.loc[correlaciones_bootstrap.variable == 'Pressure3pm', 'correlacion_pearson_s1'].iloc[0]
humidity_r = correlaciones_bootstrap.loc[correlaciones_bootstrap.variable == 'Humidity3pm', 'correlacion_pearson_s1'].iloc[0]
raintoday_r = correlaciones_bootstrap.loc[correlaciones_bootstrap.variable == 'RainToday_bin', 'correlacion_pearson_s1'].iloc[0]
max_localidad = sensibilidad_localidad.iloc[0] if not sensibilidad_localidad.empty else None

resultados_validados = pd.DataFrame([
    {
        'tipo_insumo_s3': 'Parámetro robustamente estimado',
        'resultado_validado': 'Separación de Humidity3pm entre RainTomorrow Yes y No',
        'evidencia_sumativa2': f'Diferencia bootstrap BCa 95 % [{bootstrap_parametros.loc[bootstrap_parametros.parametro == "Diferencia de medias Humidity3pm: Yes - No", "ic_bca_95_li"].iloc[0]:.4f}, {bootstrap_parametros.loc[bootstrap_parametros.parametro == "Diferencia de medias Humidity3pm: Yes - No", "ic_bca_95_ls"].iloc[0]:.4f}]',
        'decision_para_s3': 'Mantener Humidity3pm como predictor prioritario',
        'nivel_confianza': 'Alto'
    },
    {
        'tipo_insumo_s3': 'Prueba de hipótesis validada',
        'resultado_validado': 'Prueba de hipótesis sobre Humidity3pm',
        'evidencia_sumativa2': f'p permutación unilateral = {p_perm_one_sided:.6f}; {formato_p_value(p_perm_one_sided)}',
        'decision_para_s3': 'Usar la diferencia de humedad como relación estadística validada',
        'nivel_confianza': 'Alto'
    },
    {
        'tipo_insumo_s3': 'Correlación estable',
        'resultado_validado': 'Correlación Humidity3pm con RainTomorrow_bin',
        'evidencia_sumativa2': f'r = {humidity_r:.4f}',
        'decision_para_s3': 'Priorizar en modelos explicativos y predictivos',
        'nivel_confianza': 'Alto'
    },
    {
        'tipo_insumo_s3': 'Correlación estable',
        'resultado_validado': 'RainToday_bin como señal antecedente',
        'evidencia_sumativa2': f'r = {raintoday_r:.4f}',
        'decision_para_s3': 'Incluir como variable categórica binaria',
        'nivel_confianza': 'Medio-alto'
    },
    {
        'tipo_insumo_s3': 'Correlación estable',
        'resultado_validado': 'Rainfall como señal meteorológica antecedente',
        'evidencia_sumativa2': f'r = {rainfall_r:.4f}',
        'decision_para_s3': 'Incluir como variable candidata, considerando asimetría y posible transformación',
        'nivel_confianza': 'Medio'
    },
    {
        'tipo_insumo_s3': 'Correlación estable',
        'resultado_validado': 'Pressure3pm con asociación inversa',
        'evidencia_sumativa2': f'r = {pressure_r:.4f}',
        'decision_para_s3': 'Incluir por interpretación meteorológica y relación estable',
        'nivel_confianza': 'Medio'
    },
    {
        'tipo_insumo_s3': 'Robustez frente a outliers y supuestos',
        'resultado_validado': 'Robustez del contraste principal',
        'evidencia_sumativa2': f'Diferencia original = {diff_yes_no:.4f}; diferencia winsorizada = {robustez.loc[robustez.escenario == "Winsorización 1 %-99 % por grupo", "diferencia_estimador_yes_no"].iloc[0]:.4f}; diferencia de medianas = {median_yes - median_no:.4f}',
        'decision_para_s3': 'No se requiere descartar el hallazgo principal por sensibilidad a extremos',
        'nivel_confianza': 'Alto'
    },
    {
        'tipo_insumo_s3': 'Observaciones o subconjuntos influyentes',
        'resultado_validado': 'Sensibilidad por exclusión de localidad',
        'evidencia_sumativa2': 'No hay localidad evaluable' if max_localidad is None else f'Mayor variación al excluir {max_localidad["localidad_excluida"]}: {max_localidad["variacion_respecto_original"]:.4f}',
        'decision_para_s3': 'Controlar Location como variable categórica o mediante validación estratificada si se incorpora al modelo',
        'nivel_confianza': 'Medio-alto'
    }
])

resultados_validados.to_csv(DATA_PROCESSED / 'resultados_validados_sumativa2.csv', index=False)
resultados_validados.to_csv(TABLES_DIR / '10_resultados_validados_para_s3.csv', index=False)

inventario = []
for path in sorted(TABLES_DIR.glob('*.csv')):
    inventario.append({'tipo': 'tabla', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})
for path in sorted(FIGURES_DIR.glob('*.png')):
    inventario.append({'tipo': 'figura', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})
for path in sorted(DATA_PROCESSED.glob('*.csv')):
    inventario.append({'tipo': 'datos_procesados', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})

inventario_outputs = pd.DataFrame(inventario)
inventario_outputs.to_csv(SEMANA2_DIR / 'docs' / 'inventario_outputs_sumativa2.csv', index=False)

resultados_validados


,tipo_insumo_s3,resultado_validado,evidencia_sumativa2,decision_para_s3,nivel_confianza
0,Parámetro robustamente estimado,Separación de Humidity3pm entre RainTomorrow Y...,"Diferencia bootstrap BCa 95 % [22.0584, 22.5317]",Mantener Humidity3pm como predictor prioritario,Alto
1,Prueba de hipótesis validada,Prueba de hipótesis sobre Humidity3pm,"p permutación unilateral = 0.000100; p < 0,001",Usar la diferencia de humedad como relación es...,Alto
2,Correlación estable,Correlación Humidity3pm con RainTomorrow_bin,r = 0.4462,Priorizar en modelos explicativos y predictivos,Alto
3,Correlación estable,RainToday_bin como señal antecedente,r = 0.3131,Incluir como variable categórica binaria,Medio-alto
4,Correlación estable,Rainfall como señal meteorológica antecedente,r = 0.2390,"Incluir como variable candidata, considerando ...",Medio
5,Correlación estable,Pressure3pm con asociación inversa,r = -0.2260,Incluir por interpretación meteorológica y rel...,Medio
6,Robustez frente a outliers y supuestos,Robustez del contraste principal,Diferencia original = 22.2894; diferencia wins...,No se requiere descartar el hallazgo principal...,Alto
7,Observaciones o subconjuntos influyentes,Sensibilidad por exclusión de localidad,Mayor variación al excluir AliceSprings: -0.5548,Controlar Location como variable categórica o ...,Medio-alto


In [12]:
expected_outputs = [
    'data/processed/weatherAUS_sumativa2_base_validacion.csv',
    'data/processed/resultados_validados_sumativa2.csv',
    'docs/tables/00_resumen_base_sumativa2.csv',
    'docs/tables/00b_auditoria_nan_sumativa2.csv',
    'docs/tables/00c_criterio_tratamiento_nan_sumativa2.csv',
    'docs/tables/00d_resumen_casos_validos_por_analisis.csv',
    'docs/tables/00e_configuracion_remuestreo_sumativa2.csv',
    'docs/tables/01_parametros_s1_utilizados.csv',
    'docs/tables/02_bootstrap_parametros_s1.csv',
    'docs/tables/02b_comparacion_intervalos_bootstrap.csv',
    'docs/tables/03_test_permutacion_humidity3pm.csv',
    'docs/tables/04_bootstrap_correlaciones.csv',
    'docs/tables/04b_diagnostico_estabilidad_correlaciones.csv',
    'docs/tables/05_resultados_montecarlo_resumen.csv',
    'docs/tables/06_resultados_montecarlo_umbrales.csv',
    'docs/tables/07_convergencia_montecarlo.csv',
    'docs/tables/08_robustez_outliers_supuestos.csv',
    'docs/tables/09_sensibilidad_por_localidad.csv',
    'docs/tables/09b_sintesis_robustez.csv',
    'docs/tables/10_resultados_validados_para_s3.csv',
    'docs/figures/fig_01_bootstrap_media_global_humidity3pm.png',
    'docs/figures/fig_02_bootstrap_media_humidity3pm_no.png',
    'docs/figures/fig_03_bootstrap_media_humidity3pm_yes.png',
    'docs/figures/fig_04_bootstrap_diferencia_medias.png',
    'docs/figures/fig_05_bootstrap_proporcion_raintomorrow_yes.png',
    'docs/figures/fig_06_permutacion_diferencia_medias.png',
    'docs/figures/fig_07_bootstrap_correlaciones.png',
    'docs/figures/fig_07b_distribuciones_bootstrap_correlaciones.png',
    'docs/figures/fig_08_montecarlo_distribucion_delta.png',
    'docs/figures/fig_09_convergencia_montecarlo.png',
    'docs/figures/fig_10_robustez_diferencia_medias.png',
    'docs/inventario_outputs_sumativa2.csv'
]

control_integridad = []
for rel in expected_outputs:
    path = SEMANA2_DIR / rel
    control_integridad.append({
        'salida_esperada': rel,
        'existe': path.exists(),
        'tamano_bytes': int(path.stat().st_size) if path.exists() else 0
    })

control_integridad = pd.DataFrame(control_integridad)
control_integridad.to_csv(TABLES_DIR / '11_control_integridad_salidas_sumativa2.csv', index=False)

inventario = []
for path in sorted(TABLES_DIR.glob('*.csv')):
    inventario.append({'tipo': 'tabla', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})
for path in sorted(FIGURES_DIR.glob('*.png')):
    inventario.append({'tipo': 'figura', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})
for path in sorted(DATA_PROCESSED.glob('*.csv')):
    inventario.append({'tipo': 'datos_procesados', 'archivo': path.relative_to(SEMANA2_DIR).as_posix(), 'descripcion': path.stem})

inventario_outputs = pd.DataFrame(inventario)
inventario_outputs.to_csv(SEMANA2_DIR / 'docs' / 'inventario_outputs_sumativa2.csv', index=False)

salidas_faltantes = control_integridad.loc[~control_integridad['existe'], 'salida_esperada'].tolist()
if salidas_faltantes:
    raise FileNotFoundError(f'Salidas esperadas no generadas: {salidas_faltantes}')

control_integridad


,salida_esperada,existe,tamano_bytes
0,data/processed/weatherAUS_sumativa2_base_valid...,True,11343047
1,data/processed/resultados_validados_sumativa2.csv,True,1543
2,docs/tables/00_resumen_base_sumativa2.csv,True,240
3,docs/tables/00b_auditoria_nan_sumativa2.csv,True,695
4,docs/tables/00c_criterio_tratamiento_nan_sumat...,True,1461
5,docs/tables/00d_resumen_casos_validos_por_anal...,True,697
6,docs/tables/00e_configuracion_remuestreo_sumat...,True,527
7,docs/tables/01_parametros_s1_utilizados.csv,True,501
8,docs/tables/02_bootstrap_parametros_s1.csv,True,1585
9,docs/tables/02b_comparacion_intervalos_bootstr...,True,2515


## 10. Control metodológico final

La Sumativa 2 valida computacionalmente los resultados obtenidos en la Sumativa 1, utilizando como base weatherAUS_sumativa1_variables_clave.csv. El análisis se centra en la relación entre Humidity3pm y RainTomorrow, dado que en la etapa anterior se identificó una diferencia relevante de humedad entre los días con lluvia y sin lluvia al día siguiente. Los valores faltantes se trataron mediante casos válidos por análisis, sin imputación en esta fase. Esta decisión permite evaluar los estimadores directamente sobre datos observados y reservar la imputación para la etapa predictiva de la Sumativa 3.

Los intervalos clásicos de la Sumativa 1 fueron contrastados mediante bootstrap no paramétrico con 10.000 remuestras, incorporando intervalos percentil y BCa. La diferencia de medias de Humidity3pm fue validada además mediante una prueba de permutación con 10.000 permutaciones, lo que permite contrastar el resultado principal sin depender exclusivamente de supuestos paramétricos.

Las correlaciones relevantes con RainTomorrow_bin fueron evaluadas mediante intervalos bootstrap al 95 %, identificando variables estables para la fase posterior, principalmente Humidity3pm, RainToday_bin, Rainfall y Pressure3pm.

La simulación Monte Carlo se construyó con parámetros estimados desde la Sumativa 1 y permitió evaluar escenarios de humedad y umbrales asociados a la ocurrencia de lluvia al día siguiente. Complementariamente, el análisis de robustez mostró que la diferencia entre grupos se mantiene al aplicar filtro IQR, winsorización, diferencia de medianas y sensibilidad por localidad. En conjunto, los resultados respaldan que Humidity3pm es una variable prioritaria para la Sumativa 3. En esa etapa, cualquier imputación o transformación deberá ajustarse solo con los datos de entrenamiento, evitando fuga de información hacia validación o prueba.


## 11. Conclusión técnica del notebook

La validación computacional confirma que el principal resultado de la Sumativa 1 es estable: `Humidity3pm` mantiene una diferencia positiva, amplia y robusta entre días con y sin lluvia al día siguiente. Los intervalos bootstrap percentil y BCa son concordantes con los intervalos clásicos, la prueba de permutación respalda la significancia de la diferencia de medias, y las correlaciones de mayor magnitud conservan su dirección bajo remuestreo. Las distribuciones bootstrap de correlación permiten verificar visualmente que los intervalos relevantes se mantienen alejados del cero.

La simulación Monte Carlo traduce esta relación a escenarios probabilísticos individuales y muestra que umbrales altos de humedad aumentan la probabilidad condicional de lluvia al día siguiente. Esta simulación no reemplaza el análisis inferencial ni constituye un modelo predictivo final. Para la Sumativa 3, `Humidity3pm`, `RainToday_bin`, `Rainfall` y `Pressure3pm` deben considerarse variables candidatas prioritarias, integrando validación cruzada, tratamiento de faltantes dentro del flujo de entrenamiento y métricas de clasificación para controlar sobreajuste y desempeño predictivo.
